# Demand Forecasting: Iowa Liquor Sales
**Studi Kasus:** Prediksi Permintaan Pasar (Market Demand Prediction)

**Konteks Bisnis:** Pemerintah Negara Bagian Iowa bertindak sebagai distributor tunggal (wholesaler) minuman keras.
Model prediktif ini dirancang untuk membantu manajemen logistik gudang pusat dalam mengoptimalkan stok -- mencegah *overstock* yang mengikat modal dan *understock* yang menyebabkan kehilangan penjualan.

**Stack Teknologi:** PySpark (pyspark.sql + pyspark.ml) sebagai Big Data processing framework.

**Dataset:** Iowa Liquor Sales (~37.7 juta transaksi, ~3.3 GB)

## Langkah 1: Inisialisasi Spark Session & Data Cleaning

### 1.1 Inisialisasi SparkSession
Konfigurasi memori disesuaikan untuk menangani dataset ~3.3 GB secara lokal.

In [1]:
import os
import socket
import time
from pyspark.sql import SparkSession

# --- 1. KONFIGURASI JARINGAN & PATH ---
MASTER_IP   = "192.168.1.31" # Sesuaikan dengan IP laptop i7
MASTER_PORT = 7077
MASTER_URL  = f"spark://{MASTER_IP}:{MASTER_PORT}"

DRIVER_IP          = "192.168.1.31" # Sesuaikan dengan IP laptop i7
DRIVER_PORT        = 40400
BLOCK_MANAGER_PORT = 40401

# Pastikan path ini SAMA PERSIS di laptop i3, i5, dan i7
VENV_PYTHON = r"d:\Kuliah\BigData2\Final-project\venv\Scripts\python.exe"
os.environ["PYSPARK_PYTHON"]        = VENV_PYTHON
os.environ["PYSPARK_DRIVER_PYTHON"] = VENV_PYTHON

# --- 2. RESOURCE TUNING ---
EXECUTOR_CORES  = 3
EXECUTOR_MEMORY = "3g"
MIN_WORKERS     = 2 # Tunggu minimal 2 laptop nyala sebelum mulai

print(f"Menghubungkan ke Cluster: {MASTER_URL}...")

spark = (
    SparkSession.builder
    .appName('Iowa_Liquor_Demand_Forecasting_Cluster')
    .master(MASTER_URL)
    .config("spark.driver.host", DRIVER_IP)
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.port", str(DRIVER_PORT))
    .config("spark.blockManager.port", str(BLOCK_MANAGER_PORT))
    
    # Alokasi Resource
    .config("spark.driver.memory", "4g") # Driver butuh RAM untuk .toPandas() di EDA
    .config("spark.executor.cores", str(EXECUTOR_CORES))
    .config("spark.executor.memory", EXECUTOR_MEMORY)
    
    # Tuning Performa Network & Shuffle
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.network.timeout", "300s") # Cegah timeout saat load 3GB file
    .config("spark.executor.heartbeatInterval", "60s")
    .getOrCreate()
)

sc = spark.sparkContext
sc.setLogLevel('WARN')

# --- 3. ADAPTIVE WORKER WAITING LOGIC ---
print(f"Menunggu minimal {MIN_WORKERS} executor terhubung...")
active_executors = 0
for attempt in range(30):
    mem_status = sc._jsc.sc().getExecutorMemoryStatus()
    active_executors = max(0, mem_status.size() - 1)
    
    if active_executors >= MIN_WORKERS:
        print(f"{active_executors} Executor aktif! Cluster siap.")
        break
    time.sleep(3)
else:
    raise RuntimeError(f"Timeout: Hanya {active_executors} executor aktif.")

# --- 4. DYNAMIC SHUFFLE PARTITIONS ---
# Mengatur partisi shuffle berdasarkan total core yang sedang aktif
total_cores_active = active_executors * EXECUTOR_CORES
optimal_partitions = total_cores_active * 3 # 3 partisi per core adalah sweet spot
spark.conf.set("spark.sql.shuffle.partitions", str(optimal_partitions))

print(f'Spark Version      : {spark.version}')
print(f'Total Active Cores : {total_cores_active}')
print(f'Shuffle Partitions : {optimal_partitions}')

Menghubungkan ke Cluster: spark://192.168.1.31:7077...
Menunggu minimal 2 executor terhubung...
4 Executor aktif! Cluster siap.
Spark Version      : 4.1.1
Total Active Cores : 12
Shuffle Partitions : 36


### 1.2 Memuat Dataset

In [2]:
raw_df = (
    spark.read.csv(
        'Iowa_Liquor_Sales.csv',
        header=True,
        inferSchema=True,
        escape='"',
        # multiLine=True diperlukan jika ada field CSV yang mengandung newline.
        # Catatan performa: opsi ini menonaktifkan parallelism (single partition).
        # Jika CSV bersih, hapus opsi ini untuk loading lebih cepat.
        multiLine=True
    )
)

print(f'Jumlah baris: {raw_df.count():,}')
print(f'Jumlah kolom: {len(raw_df.columns)}')
raw_df.printSchema()

Jumlah baris: 12,591,077
Jumlah kolom: 24
root
 |-- Invoice/Item Number: string (nullable = true)
 |-- Date: string (nullable = true)
 |-- Store Number: integer (nullable = true)
 |-- Store Name: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Zip Code: string (nullable = true)
 |-- Store Location: string (nullable = true)
 |-- County Number: integer (nullable = true)
 |-- County: string (nullable = true)
 |-- Category: integer (nullable = true)
 |-- Category Name: string (nullable = true)
 |-- Vendor Number: integer (nullable = true)
 |-- Vendor Name: string (nullable = true)
 |-- Item Number: integer (nullable = true)
 |-- Item Description: string (nullable = true)
 |-- Pack: integer (nullable = true)
 |-- Bottle Volume (ml): integer (nullable = true)
 |-- State Bottle Cost: string (nullable = true)
 |-- State Bottle Retail: string (nullable = true)
 |-- Bottles Sold: integer (nullable = true)
 |-- Sale (Dollars): string (nulla

In [3]:
raw_df.show(5, truncate=30)

+-------------------+----------+------------+------------------------------+-------------------+------------+--------+------------------------------+-------------+-------+--------+-------------+-------------+---------------------------+-----------+------------------------------+----+------------------+-----------------+-------------------+------------+--------------+--------------------+---------------------+
|Invoice/Item Number|      Date|Store Number|                    Store Name|            Address|        City|Zip Code|                Store Location|County Number| County|Category|Category Name|Vendor Number|                Vendor Name|Item Number|              Item Description|Pack|Bottle Volume (ml)|State Bottle Cost|State Bottle Retail|Bottles Sold|Sale (Dollars)|Volume Sold (Liters)|Volume Sold (Gallons)|
+-------------------+----------+------------+------------------------------+-------------------+------------+--------+------------------------------+-------------+-------+---

### 1.3 Data Cleaning & Pre-processing

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType, DoubleType, IntegerType

# Standarisasi nama kolom agar mudah diakses
clean_col_names = {
    'Invoice/Item Number': 'invoice_id',
    'Date': 'date_raw',
    'Store Number': 'store_id',
    'Store Name': 'store_name',
    'Address': 'address',
    'City': 'city',
    'Zip Code': 'zip_code',
    'Store Location': 'store_location',
    'County Number': 'county_id',
    'County': 'county',
    'Category': 'category_code',
    'Category Name': 'category_name',
    'Vendor Number': 'vendor_id',
    'Vendor Name': 'vendor_name',
    'Item Number': 'item_id',
    'Item Description': 'item_desc',
    'Pack': 'pack',
    'Bottle Volume (ml)': 'bottle_volume_ml',
    'State Bottle Cost': 'bottle_cost',
    'State Bottle Retail': 'bottle_retail',
    'Bottles Sold': 'bottles_sold',
    'Sale (Dollars)': 'sale_dollars',
    'Volume Sold (Liters)': 'volume_sold_liters',
    'Volume Sold (Gallons)': 'volume_sold_gallons'
}

df = raw_df
for old_name, new_name in clean_col_names.items():
    df = df.withColumnRenamed(old_name, new_name)

df.columns

['invoice_id',
 'date_raw',
 'store_id',
 'store_name',
 'address',
 'city',
 'zip_code',
 'store_location',
 'county_id',
 'county',
 'category_code',
 'category_name',
 'vendor_id',
 'vendor_name',
 'item_id',
 'item_desc',
 'pack',
 'bottle_volume_ml',
 'bottle_cost',
 'bottle_retail',
 'bottles_sold',
 'sale_dollars',
 'volume_sold_liters',
 'volume_sold_gallons']

In [5]:
# Parsing tanggal dari format MM/DD/YYYY ke DateType
df = df.withColumn('date', F.to_date(F.col('date_raw'), 'MM/dd/yyyy'))

# Casting kolom numerik -- hapus simbol $ pada kolom harga
df = (
    df
    .withColumn('bottle_cost', F.regexp_replace('bottle_cost', '[$,]', '').cast(DoubleType()))
    .withColumn('bottle_retail', F.regexp_replace('bottle_retail', '[$,]', '').cast(DoubleType()))
    .withColumn('sale_dollars', F.regexp_replace('sale_dollars', '[$,]', '').cast(DoubleType()))
    .withColumn('volume_sold_liters', F.col('volume_sold_liters').cast(DoubleType()))
    .withColumn('volume_sold_gallons', F.col('volume_sold_gallons').cast(DoubleType()))
    .withColumn('bottles_sold', F.col('bottles_sold').cast(IntegerType()))
    .withColumn('bottle_volume_ml', F.col('bottle_volume_ml').cast(IntegerType()))
    .withColumn('pack', F.col('pack').cast(IntegerType()))
)

df.select('date', 'bottle_cost', 'bottle_retail', 'sale_dollars', 'volume_sold_liters').show(5)

+----------+-----------+-------------+------------+------------------+
|      date|bottle_cost|bottle_retail|sale_dollars|volume_sold_liters|
+----------+-----------+-------------+------------+------------------+
|2015-11-20|      18.09|        27.14|      162.84|               4.5|
|2015-11-21|      18.09|        27.14|      325.68|               9.0|
|2015-11-16|        6.4|          9.6|        19.2|               0.3|
|2015-11-04|      35.55|        53.34|      160.02|              5.25|
|2015-11-17|        6.4|          9.6|        19.2|               0.3|
+----------+-----------+-------------+------------+------------------+
only showing top 5 rows


In [6]:
# Cek missing values pada kolom-kolom kritis
critical_cols = ['date', 'category_name', 'volume_sold_liters', 'sale_dollars', 'bottles_sold']

missing_report = df.select(
    [F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in critical_cols]
)
missing_report.show()

+----+-------------+------------------+------------+------------+
|date|category_name|volume_sold_liters|sale_dollars|bottles_sold|
+----+-------------+------------------+------------+------------+
|   0|        16086|                 0|          10|           0|
+----+-------------+------------------+------------+------------+



In [7]:
# Hapus baris dengan nilai null pada kolom target dan kolom kunci
before_count = df.count()
df = df.dropna(subset=['date', 'volume_sold_liters', 'sale_dollars', 'category_name'])
after_count = df.count()
print(f'Baris sebelum cleaning: {before_count:,}')
print(f'Baris setelah cleaning:  {after_count:,}')
print(f'Baris dihapus: {before_count - after_count:,}')

Baris sebelum cleaning: 12,591,077
Baris setelah cleaning:  12,574,981
Baris dihapus: 16,096


In [8]:
# Hitung anomali DAN filter dalam satu pass menggunakan agg
# Menghindari 3x full scan: count(), count(), count()
from pyspark.sql import functions as F

anomaly_stats = df.agg(
    F.count('*').alias('total'),
    F.sum(
        F.when((F.col('volume_sold_liters') <= 0) | (F.col('sale_dollars') <= 0), 1)
        .otherwise(0)
    ).alias('anomaly_count')
).collect()[0]

print(f'Baris dengan volume/sales <= 0: {anomaly_stats["anomaly_count"]:,}')
print(f'Total sebelum filter anomali:   {anomaly_stats["total"]:,}')

df = df.filter(
    (F.col('volume_sold_liters') > 0) & (F.col('sale_dollars') > 0)
)
print(f'Baris final setelah filter anomali: {df.count():,}')


Baris dengan volume/sales <= 0: 4,309
Total sebelum filter anomali:   12,574,981
Baris final setelah filter anomali: 12,570,672


### 1.4 Binning Kategori Produk ke Super-Category
Mengelompokkan ~70 sub-kategori menjadi kategori utama untuk mengurangi noise dan memudahkan interpretasi.

In [9]:
df = df.withColumn('category_upper', F.upper(F.trim(F.col('category_name'))))

df = df.withColumn(
    'super_category',
    F.when(F.col('category_upper').like('%VODKA%'), 'VODKA')
    .when(F.col('category_upper').like('%BOURBON%'), 'BOURBON')
    .when(F.col('category_upper').like('%SCOTCH%'), 'SCOTCH')
    .when(F.col('category_upper').like('%TENNESSEE%'), 'TENNESSEE WHISKEY')
    .when(F.col('category_upper').like('%IRISH WHISK%'), 'IRISH WHISKEY')
    .when(F.col('category_upper').like('%RYE WHISK%'), 'RYE WHISKEY')
    .when(F.col('category_upper').like('%BLENDED WHISK%'), 'BLENDED WHISKEY')
    .when(F.col('category_upper').like('%CANADIAN WHISK%'), 'CANADIAN WHISKY')
    .when(F.col('category_upper').like('%WHISKEY LIQUEUR%'), 'WHISKEY LIQUEUR')
    .when(F.col('category_upper').like('%RUM%'), 'RUM')
    .when(F.col('category_upper').like('%TEQUILA%'), 'TEQUILA')
    .when(F.col('category_upper').like('%MEZCAL%'), 'TEQUILA')
    .when(F.col('category_upper').like('%GIN%'), 'GIN')
    .when(F.col('category_upper').like('%BRANDY%') | F.col('category_upper').like('%BRANDIES%'), 'BRANDY')
    .when(F.col('category_upper').like('%SCHNAPPS%'), 'SCHNAPPS')
    .when(F.col('category_upper').like('%LIQUEUR%') | F.col('category_upper').like('%CORDIAL%'), 'LIQUEUR/CORDIAL')
    .when(F.col('category_upper').like('%COCKTAIL%') | F.col('category_upper').like('%SPIRIT%'), 'COCKTAIL/SPIRIT')
    .when(F.col('category_upper').like('%CREME%'), 'LIQUEUR/CORDIAL')
    .when(F.col('category_upper').like('%AMARETTO%'), 'LIQUEUR/CORDIAL')
    .when(F.col('category_upper').like('%TRIPLE SEC%'), 'LIQUEUR/CORDIAL')
    .when(F.col('category_upper').like('%ANISETTE%'), 'LIQUEUR/CORDIAL')
    .when(F.col('category_upper').like('%COFFEE%'), 'LIQUEUR/CORDIAL')
    .when(F.col('category_upper').like('%CREAM%'), 'LIQUEUR/CORDIAL')
    .when(F.col('category_upper').like('%ROCK%RYE%'), 'LIQUEUR/CORDIAL')
    .when(F.col('category_upper').like('%ALCOHOL%'), 'NEUTRAL ALCOHOL')
    .otherwise('OTHER')
)

df.groupBy('super_category').count().orderBy(F.desc('count')).show(20, truncate=False)

+-----------------+-------+
|super_category   |count  |
+-----------------+-------+
|VODKA            |3244641|
|RUM              |1538656|
|CANADIAN WHISKY  |1207559|
|LIQUEUR/CORDIAL  |1037452|
|BOURBON          |723268 |
|BRANDY           |679517 |
|SCHNAPPS         |668965 |
|TEQUILA          |563248 |
|BLENDED WHISKEY  |554287 |
|GIN              |481169 |
|WHISKEY LIQUEUR  |453986 |
|COCKTAIL/SPIRIT  |429067 |
|SCOTCH           |387546 |
|TENNESSEE WHISKEY|317413 |
|IRISH WHISKEY    |118177 |
|RYE WHISKEY      |74292  |
|OTHER            |67078  |
|NEUTRAL ALCOHOL  |24351  |
+-----------------+-------+



In [10]:
# Ekstrak fitur waktu dasar
df = (
    df
    .withColumn('year', F.year('date'))
    .withColumn('month', F.month('date'))
    .withColumn('quarter', F.quarter('date'))
)

# Verifikasi rentang waktu
df.select(F.min('date').alias('start'), F.max('date').alias('end')).show()

+----------+----------+
|     start|       end|
+----------+----------+
|2012-01-03|2017-10-31|
+----------+----------+



In [11]:
df = df.drop('date_raw', 'category_upper')
print(f'Kolom final: {df.columns}')
df.cache()
print(f'Total records setelah cleaning: {df.count():,}')

Kolom final: ['invoice_id', 'store_id', 'store_name', 'address', 'city', 'zip_code', 'store_location', 'county_id', 'county', 'category_code', 'category_name', 'vendor_id', 'vendor_name', 'item_id', 'item_desc', 'pack', 'bottle_volume_ml', 'bottle_cost', 'bottle_retail', 'bottles_sold', 'sale_dollars', 'volume_sold_liters', 'volume_sold_gallons', 'date', 'super_category', 'year', 'month', 'quarter']
Total records setelah cleaning: 12,570,672


## Langkah 2: Exploratory Data Analysis (EDA)

### 2.1 Tren Volume Tahunan

In [12]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style='whitegrid', palette='deep')
plt.rcParams['figure.dpi'] = 120


In [13]:
yearly_vol = (
    df.groupBy('year')
    .agg(F.sum('volume_sold_liters').alias('total_volume_liters'))
    .orderBy('year')
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(yearly_vol['year'].astype(str), yearly_vol['total_volume_liters'] / 1e6,
       color=sns.color_palette('Blues_d', len(yearly_vol)), edgecolor='black', linewidth=0.5)
ax.set_xlabel('Tahun')
ax.set_ylabel('Volume Terjual (Juta Liter)')
ax.set_title('Total Volume Penjualan Minuman Keras Iowa per Tahun')
for i, v in enumerate(yearly_vol['total_volume_liters']):
    ax.text(i, v/1e6 + 0.2, f'{v/1e6:.1f}M', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('eda_yearly_volume.png', bbox_inches='tight')
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_5956\3959905026.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** Volume penjualan menunjukkan tren pertumbuhan yang konsisten dari tahun ke tahun. 
Pertumbuhan ini penting diperhitungkan dalam model sebagai tren dasar agar prediksi tidak *underestimate* permintaan di tahun-tahun mendatang.

### 2.2 Tren Volume Bulanan (Pola Musiman)

In [14]:
monthly_vol = (
    df.groupBy('year', 'month')
    .agg(F.sum('volume_sold_liters').alias('total_volume'))
    .orderBy('year', 'month')
    .toPandas()
)
monthly_vol['period'] = monthly_vol['year'].astype(str) + '-' + monthly_vol['month'].astype(str).str.zfill(2)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(monthly_vol['period'], monthly_vol['total_volume'] / 1e6, marker='o', markersize=3, linewidth=1.2)
ax.set_xlabel('Periode (Tahun-Bulan)')
ax.set_ylabel('Volume (Juta Liter)')
ax.set_title('Tren Volume Penjualan Bulanan')
ax.tick_params(axis='x', rotation=90, labelsize=7)
plt.tight_layout()
plt.savefig('eda_monthly_trend.png', bbox_inches='tight')
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_5956\2723013275.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# Seasonality pattern: rata-rata volume per bulan di semua tahun
seasonal = (
    df.groupBy('month')
    .agg(F.avg('volume_sold_liters').alias('avg_volume_per_txn'),
         F.sum('volume_sold_liters').alias('total_volume'))
    .orderBy('month')
    .toPandas()
)

fig, ax = plt.subplots(figsize=(10, 5))
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
ax.bar(seasonal['month'], seasonal['total_volume'] / 1e6, color=sns.color_palette('coolwarm', 12), edgecolor='black', linewidth=0.4)
ax.set_xticks(range(1,13))
ax.set_xticklabels(month_labels)
ax.set_ylabel('Total Volume (Juta Liter)')
ax.set_title('Pola Musiman: Distribusi Volume per Bulan (Semua Tahun)')
plt.tight_layout()
plt.savefig('eda_seasonality.png', bbox_inches='tight')
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_5956\665623427.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** Pola musiman yang jelas terlihat -- volume penjualan cenderung meningkat menjelang akhir tahun (Oktober-Desember), 
kemungkinan didorong oleh *holiday season* (Thanksgiving, Natal, Tahun Baru). 
Bulan-bulan awal tahun (Januari-Februari) menunjukkan penurunan. 
Pola ini menjadi dasar pentingnya fitur `month` dan `quarter` dalam model prediktif.

### 2.3 Analisis Pangsa Pasar per Super-Category

In [16]:
cat_vol = (
    df.groupBy('super_category')
    .agg(F.sum('volume_sold_liters').alias('total_volume'))
    .orderBy(F.desc('total_volume'))
    .toPandas()
)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Bar chart
axes[0].barh(cat_vol['super_category'][::-1], cat_vol['total_volume'][::-1] / 1e6,
             color=sns.color_palette('viridis', len(cat_vol)))
axes[0].set_xlabel('Volume (Juta Liter)')
axes[0].set_title('Volume Penjualan per Super-Category')

# Pie chart -- top 8 + Others
top_n = 8
pie_data = cat_vol.head(top_n).copy()
others_vol = cat_vol.iloc[top_n:]['total_volume'].sum()
import pandas as pd
pie_data = pd.concat([pie_data, pd.DataFrame([{'super_category': 'OTHERS', 'total_volume': others_vol}])], ignore_index=True)
axes[1].pie(pie_data['total_volume'], labels=pie_data['super_category'],
            autopct='%1.1f%%', startangle=140, textprops={'fontsize': 8})
axes[1].set_title('Pangsa Pasar Volume (Top 8 + Others)')

plt.tight_layout()
plt.savefig('eda_category_share.png', bbox_inches='tight')
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_5956\595278417.py:28: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** VODKA mendominasi pasar minuman keras Iowa dengan pangsa volume terbesar, diikuti WHISKEY variants dan RUM. 
Dominasi ini berarti *forecast error* pada kategori VODKA akan memiliki dampak logistik terbesar terhadap gudang. 
Model perlu memberikan perhatian ekstra pada akurasi prediksi kategori-kategori bervolume tinggi.

### 2.4 Analisis Korelasi Harga vs Volume

In [17]:
from pyspark.ml.stat import Correlation
from pyspark.ml.feature import VectorAssembler

# Hitung rata-rata harga per liter di level transaksi
corr_df = df.withColumn(
    'price_per_liter',
    F.col('sale_dollars') / F.col('volume_sold_liters')
).filter(F.col('price_per_liter').isNotNull() & F.col('price_per_liter').between(1, 500))

# Agregasi per kategori-bulan untuk korelasi yang lebih bermakna
corr_agg = (
    corr_df.groupBy('super_category', 'year', 'month')
    .agg(
        F.avg('price_per_liter').alias('avg_price_per_liter'),
        F.sum('volume_sold_liters').alias('total_volume')
    )
)

assembler = VectorAssembler(
    inputCols=['avg_price_per_liter', 'total_volume'],
    outputCol='features'
)
corr_vector = assembler.transform(corr_agg).select('features')

corr_matrix = Correlation.corr(corr_vector, 'features', 'pearson').head()[0]
print('Pearson Correlation Matrix:')
print(f'  Price vs Volume: {corr_matrix.toArray()[0][1]:.4f}')

Pearson Correlation Matrix:
  Price vs Volume: -0.3366


In [18]:
corr_pd = corr_agg.toPandas()

fig, ax = plt.subplots(figsize=(10, 6))
scatter = ax.scatter(
    corr_pd['avg_price_per_liter'],
    corr_pd['total_volume'] / 1e3,
    alpha=0.4, s=20, c=corr_pd['avg_price_per_liter'],
    cmap='RdYlGn_r', edgecolors='none'
)
ax.set_xlabel('Rata-rata Harga per Liter ($)')
ax.set_ylabel('Total Volume (Ribu Liter)')
ax.set_title('Hubungan Harga per Liter vs Volume Terjual (per Kategori-Bulan)')
plt.colorbar(scatter, label='Harga/Liter ($)')
plt.tight_layout()
plt.savefig('eda_price_volume.png', bbox_inches='tight')
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_5956\53732167.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** Korelasi negatif antara harga per liter dan volume penjualan mengindikasikan pola ekonomi yang rasional -- 
produk dengan harga lebih terjangkau terjual dalam volume lebih besar (VODKA, RUM), sementara kategori premium 
(SCOTCH, SINGLE MALT) terjual dalam volume lebih rendah. Fitur harga rata-rata menjadi prediktor yang relevan 
karena mencerminkan elastisitas permintaan per kategori.

## Langkah 3: Agregasi Data & Pembentukan Variabel Target (Y)

Granularitas data diubah dari transaksi individual menjadi **bulanan per Super-Category**. 
Variabel target (Y) = Total Volume Terjual (Liter) per kategori per bulan.

In [19]:
monthly_agg = (
    df.groupBy('super_category', 'year', 'month')
    .agg(
        F.sum('volume_sold_liters').alias('total_volume_liters'),
        F.sum('sale_dollars').alias('total_sales_dollars'),
        F.sum('bottles_sold').alias('total_bottles_sold'),
        # Harga per liter = total_revenue / total_volume (volume-weighted)
        # Lebih akurat daripada avg(rasio) yang memberi bobot sama tiap transaksi
        (F.sum('sale_dollars') / F.sum('volume_sold_liters')).alias('avg_price_per_liter'),
        F.countDistinct('store_id').alias('num_stores'),
        F.countDistinct('item_id').alias('num_products'),
        F.count('*').alias('num_transactions')
    )
    .orderBy('super_category', 'year', 'month')
)

print(f'Jumlah baris setelah agregasi: {monthly_agg.count()}')
print(f'Jumlah Super-Category unik: {monthly_agg.select("super_category").distinct().count()}')
monthly_agg.show(10, truncate=False)

Jumlah baris setelah agregasi: 1246
Jumlah Super-Category unik: 18
+---------------+----+-----+-------------------+-------------------+------------------+-------------------+----------+------------+----------------+
|super_category |year|month|total_volume_liters|total_sales_dollars|total_bottles_sold|avg_price_per_liter|num_stores|num_products|num_transactions|
+---------------+----+-----+-------------------+-------------------+------------------+-------------------+----------+------------+----------------+
|BLENDED WHISKEY|2012|1    |67090.79999999987  |600992.2600000001  |67793             |8.957893779773103  |826       |49          |7527            |
|BLENDED WHISKEY|2012|2    |64533.06999999984  |583145.729999997   |64942             |9.036385995583325  |849       |49          |7658            |
|BLENDED WHISKEY|2012|3    |67663.50999999991  |610149.2000000004  |68071             |9.017403915345232  |849       |53          |7555            |
|BLENDED WHISKEY|2012|4    |68507.23999

## Langkah 4: Feature Engineering

### 4.1 Fitur Waktu & Indikator Hari Libur

In [20]:
from pyspark.sql import Window

# ── Quarter ────────────────────────────────────────────────────
monthly_agg = monthly_agg.withColumn(
    'quarter',
    F.quarter(F.make_date('year', 'month', F.lit(1)))
)

# ── Indikator Bulan Hari Libur Besar AS ───────────────────────
# Jan(1) : New Year's Day
# Feb(2) : Super Bowl
# May(5) : Memorial Day
# Jul(7) : Independence Day
# Sep(9) : Labor Day
# Nov(11): Thanksgiving
# Dec(12): Christmas & New Year's Eve
holiday_months = [1, 2, 5, 7, 9, 11, 12]

monthly_agg = monthly_agg.withColumn(
    'is_holiday_month',
    F.when(F.col('month').isin(holiday_months), 1).otherwise(0)
)

# ── Indikator Peak Season (Q4) ─────────────────────────────────
# Oktober–Desember adalah puncak penjualan berdasarkan EDA
monthly_agg = monthly_agg.withColumn(
    'is_peak_season',
    F.when(F.col('month').isin([10, 11, 12]), 1).otherwise(0)
)

# ── Indikator Low Season (Q1) ──────────────────────────────────
# Januari–Februari adalah titik terendah berdasarkan EDA
monthly_agg = monthly_agg.withColumn(
    'is_low_season',
    F.when(F.col('month').isin([1, 2]), 1).otherwise(0)
)

# ── Posisi Bulan dalam Quarter ─────────────────────────────────
# Bulan ke-1, ke-2, atau ke-3 dalam quarternya
# Berguna untuk menangkap pola restocking awal quarter
monthly_agg = monthly_agg.withColumn(
    'month_in_quarter',
    ((F.col('month') - 1) % 3) + 1
)

# ── Fitur Siklus Trigonometri (Sin/Cos Encoding) ───────────────
# Menggantikan 'month' sebagai integer (1–12) yang bersifat linear
# Sin/Cos encoding menangkap bahwa Desember dan Januari itu "dekat"
# secara musiman, sesuatu yang tidak bisa ditangkap oleh integer biasa
monthly_agg = monthly_agg.withColumn(
    'month_sin',
    F.sin(2 * 3.14159265 * F.col('month') / 12)
).withColumn(
    'month_cos',
    F.cos(2 * 3.14159265 * F.col('month') / 12)
)

# Verifikasi hasil
monthly_agg.select(
    'super_category', 'year', 'month', 'quarter',
    'is_holiday_month', 'is_peak_season', 'is_low_season',
    'month_in_quarter', 'month_sin', 'month_cos'
).show(15, truncate=False)

+---------------+----+-----+-------+----------------+--------------+-------------+----------------+----------------------+----------------------+
|super_category |year|month|quarter|is_holiday_month|is_peak_season|is_low_season|month_in_quarter|month_sin             |month_cos             |
+---------------+----+-----+-------+----------------+--------------+-------------+----------------+----------------------+----------------------+
|BLENDED WHISKEY|2012|1    |1      |1               |0             |1            |1               |0.499999999481858     |0.8660254040835881    |
|BLENDED WHISKEY|2012|2    |1      |1               |0             |1            |2               |0.8660254031861399    |0.500000001036284     |
|BLENDED WHISKEY|2012|3    |1      |0               |0             |0            |3               |1.0                   |1.7948965149208059E-9 |
|BLENDED WHISKEY|2012|4    |2      |0               |0             |0            |1               |0.8660254049810362    |-0

### 4.2 Fitur Karakteristik Produk (Encoding Kategorikal)

In [21]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder

indexer = StringIndexer(
    inputCol='super_category',
    outputCol='category_index',
    handleInvalid='keep'
)

encoder = OneHotEncoder(
    inputCol='category_index',
    outputCol='category_ohe'
)

indexer_model = indexer.fit(monthly_agg)
monthly_agg = indexer_model.transform(monthly_agg)

encoder_model = encoder.fit(monthly_agg)
monthly_agg = encoder_model.transform(monthly_agg)

# Mapping index ke label untuk interpretasi
print('Category Index Mapping:')
for i, label in enumerate(indexer_model.labels):
    print(f'  {i} -> {label}')

Category Index Mapping:
  0 -> BLENDED WHISKEY
  1 -> BOURBON
  2 -> BRANDY
  3 -> CANADIAN WHISKY
  4 -> COCKTAIL/SPIRIT
  5 -> GIN
  6 -> IRISH WHISKEY
  7 -> LIQUEUR/CORDIAL
  8 -> OTHER
  9 -> RUM
  10 -> RYE WHISKEY
  11 -> SCHNAPPS
  12 -> SCOTCH
  13 -> TENNESSEE WHISKEY
  14 -> TEQUILA
  15 -> VODKA
  16 -> WHISKEY LIQUEUR
  17 -> NEUTRAL ALCOHOL


### 4.3 Fitur Historis (Lag & Rolling Average) dengan Window Functions

In [22]:
from pyspark.sql import Window

w_cat = Window.partitionBy('super_category').orderBy('year', 'month')

monthly_agg = (
    monthly_agg

    .withColumn('lag_1_month',  F.lag('total_volume_liters', 1).over(w_cat))
    .withColumn('lag_2_month',  F.lag('total_volume_liters', 2).over(w_cat))
    .withColumn('lag_3_month',  F.lag('total_volume_liters', 3).over(w_cat))

    .withColumn('lag_6_month',  F.lag('total_volume_liters', 6).over(w_cat))

    .withColumn('lag_11_month', F.lag('total_volume_liters', 11).over(w_cat))
    .withColumn('lag_12_month', F.lag('total_volume_liters', 12).over(w_cat))
    .withColumn('lag_13_month', F.lag('total_volume_liters', 13).over(w_cat))

    .withColumn('rolling_avg_3m',
        F.avg('total_volume_liters').over(w_cat.rowsBetween(-3, -1))
    )
    .withColumn('rolling_avg_6m',
        F.avg('total_volume_liters').over(w_cat.rowsBetween(-6, -1))
    )
    .withColumn('rolling_std_3m',
        F.stddev('total_volume_liters').over(w_cat.rowsBetween(-3, -1))
    )

    .withColumn('yoy_growth',
        # Gunakan lag_12_month yang sudah dihitung — hindari rekomputasi window
        # Guard division-by-zero: jika lag_12 = 0, hasilkan None (null)
        F.when(
            F.col('lag_12_month').isNotNull() & (F.col('lag_12_month') != 0),
            (F.col('total_volume_liters') - F.col('lag_12_month')) / F.col('lag_12_month')
        ).otherwise(F.lit(None).cast('double'))
    )
)

print(f'Baris sebelum drop null: {monthly_agg.count()}')

monthly_agg.select(
    'super_category', 'year', 'month',
    'total_volume_liters',
    'lag_1_month', 'lag_3_month', 'lag_6_month', 'lag_12_month',
    'rolling_avg_3m', 'rolling_avg_6m', 'rolling_std_3m',
    'yoy_growth'
).show(15, truncate=False)



Baris sebelum drop null: 1246
+---------------+----+-----+-------------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+--------------------+
|super_category |year|month|total_volume_liters|lag_1_month      |lag_3_month      |lag_6_month      |lag_12_month     |rolling_avg_3m   |rolling_avg_6m   |rolling_std_3m    |yoy_growth          |
+---------------+----+-----+-------------------+-----------------+-----------------+-----------------+-----------------+-----------------+-----------------+------------------+--------------------+
|BLENDED WHISKEY|2012|1    |67090.79999999987  |NULL             |NULL             |NULL             |NULL             |NULL             |NULL             |NULL              |NULL                |
|BLENDED WHISKEY|2012|2    |64533.06999999984  |67090.79999999987|NULL             |NULL             |NULL             |67090.79999999987|67090.79999999987|NULL              |NULL   

In [23]:
# Bersihkan nilai Infinity yang bisa terjadi di fitur derived (yoy_growth)
# sebelum dropna, agar tidak ada baris yang seharusnya valid ikut terbuang
from pyspark.sql.functions import isnan

monthly_agg = monthly_agg.replace(float('inf'),  None) \
                         .replace(float('-inf'), None)

featured_df = monthly_agg.dropna(subset=[
    'lag_1_month', 'lag_2_month', 'lag_3_month',
    'lag_6_month', 'lag_11_month', 'lag_12_month', 'lag_13_month',
    'rolling_avg_3m', 'rolling_avg_6m', 'rolling_std_3m',
    'yoy_growth'
])

print(f'Baris setelah drop null lag: {featured_df.count()}')
print(f'Baris yang dibuang: {monthly_agg.count() - featured_df.count()}')

Baris setelah drop null lag: 1012
Baris yang dibuang: 234


### 4.4 Assembling Fitur ke dalam Vektor

In [24]:
from pyspark.ml.feature import VectorAssembler

feature_cols = [
    # Fitur waktu
    'month', 'quarter',
    'month_sin', 'month_cos',       
    'month_in_quarter',             

    # Indikator musiman
    'is_holiday_month',
    'is_peak_season',               
    'is_low_season',                

    # Fitur bisnis
    'avg_price_per_liter', 'num_stores', 'num_products', 'num_transactions',

    # Lag jangka pendek
    'lag_1_month', 'lag_2_month', 'lag_3_month',

    # Lag jangka menengah & panjang
    'lag_6_month', 'lag_11_month', 'lag_12_month', 'lag_13_month',

    # Rolling statistics
    'rolling_avg_3m', 'rolling_avg_6m', 'rolling_std_3m',

    # Tren YoY
    'yoy_growth',

    # Encoding kategori
    'category_ohe'
]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol='features',
    # 'error': akan raise exception jika masih ada null/Inf lolos bersih —
    # deteksi masalah lebih baik daripada 'skip' yang silent drop baris
    handleInvalid='error'
)

model_df = assembler.transform(featured_df)
model_df = model_df.select('super_category', 'year', 'month', 'features', 'total_volume_liters')
model_df = model_df.withColumnRenamed('total_volume_liters', 'label')

print(f'Dataset siap model: {model_df.count()} baris')
model_df.show(5, truncate=40)

Dataset siap model: 1012 baris
+---------------+----+-----+----------------------------------------+------------------+
| super_category|year|month|                                features|             label|
+---------------+----+-----+----------------------------------------+------------------+
|BLENDED WHISKEY|2013|    2|(41,[0,1,2,3,4,5,7,8,9,10,11,12,13,14...| 65418.92999999989|
|BLENDED WHISKEY|2013|    3|(41,[0,1,2,3,4,8,9,10,11,12,13,14,15,...|61491.339999999895|
|BLENDED WHISKEY|2013|    4|(41,[0,1,2,3,4,8,9,10,11,12,13,14,15,...| 85266.83000000002|
|BLENDED WHISKEY|2013|    5|(41,[0,1,2,3,4,5,8,9,10,11,12,13,14,1...|  72198.1899999999|
|BLENDED WHISKEY|2013|    6|(41,[0,1,2,3,4,8,9,10,11,12,13,14,15,...|  63605.0299999999|
+---------------+----+-----+----------------------------------------+------------------+
only showing top 5 rows


## Langkah 5: Pemisahan Data & Pemodelan Regresi

### 5.1 Time-Based Train/Test Split
Data deret waktu tidak boleh di-split secara random. Kita menggunakan tahun terakhir (2017) sebagai data uji, 
dan tahun sebelumnya sebagai data latih.

In [25]:
# Identifikasi tahun terakhir dalam dataset
max_year = model_df.agg(F.max('year')).collect()[0][0]
print(f'Tahun terakhir (test set): {max_year}')

train_df = model_df.filter(F.col('year') < max_year)
test_df = model_df.filter(F.col('year') == max_year)

print(f'Training set: {train_df.count()} baris (tahun < {max_year})')
print(f'Test set:     {test_df.count()} baris (tahun = {max_year})')

Tahun terakhir (test set): 2017
Training set: 842 baris (tahun < 2017)
Test set:     170 baris (tahun = 2017)


### 5.2 Training Model: Gradient-Boosted Tree Regressor

In [26]:
from pyspark.ml.regression import GBTRegressor

gbt = GBTRegressor(
    featuresCol='features',
    labelCol='label',
    maxIter=100,
    maxDepth=6,
    stepSize=0.1,
    subsamplingRate=0.8,
    seed=42
)

print('Training GBTRegressor...')
gbt_model = gbt.fit(train_df)
print('Training selesai.')

Training GBTRegressor...
Training selesai.


### 5.3 Training Model Pembanding: Random Forest Regressor

In [27]:
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.tuning import TrainValidationSplit, ParamGridBuilder
from pyspark.ml.evaluation import RegressionEvaluator

# ── PENTING: CrossValidator dilarang pada data deret waktu ─────────────
# CrossValidator mengacak data saat k-fold → data masa depan bocor ke
# training (data leakage). Ganti dengan TrainValidationSplit berbasis
# proporsi temporal: 80% train awal, 20% validasi akhir.
# trainRatio=0.8 berarti 80% pertama data (urutan asli) dipakai sebagai train.
# ───────────────────────────────────────────────────────────────────────

rf = RandomForestRegressor(
    featuresCol='features',
    labelCol='label',
    numTrees=200,
    maxDepth=10,
    minInstancesPerNode=2,
    featureSubsetStrategy='sqrt',
    subsamplingRate=0.8,
    seed=42
)

# Grid search — 6 kombinasi (2x3)
paramGrid = (
    ParamGridBuilder()
    .addGrid(rf.numTrees, [100, 200])
    .addGrid(rf.maxDepth, [8, 10, 12])
    .build()
)

evaluator = RegressionEvaluator(
    labelCol='label',
    predictionCol='prediction',
    metricName='rmse'
)

# TrainValidationSplit: 80% data pertama (train), 20% terakhir (validasi)
# Urutan data sudah temporal (year, month) → tidak perlu shuffle
tvs = TrainValidationSplit(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    trainRatio=0.8,   # 80% awal = train, 20% akhir = validasi
    seed=42
)

print('Training RandomForest dengan TrainValidationSplit (time-safe)...')
tvs_model = tvs.fit(train_df)
rf_model  = tvs_model.bestModel
print(f'Best numTrees : {rf_model.getNumTrees}')
print(f'Best maxDepth : {rf_model.getMaxDepth()}')


Training RandomForest dengan TrainValidationSplit (time-safe)...
Best numTrees : 100
Best maxDepth : 12


## Langkah 6: Evaluasi Model

### 6.1 Prediksi pada Test Set

In [28]:
gbt_predictions = gbt_model.transform(test_df)
rf_predictions = rf_model.transform(test_df)

### 6.2 Metrik Evaluasi Statistik

In [29]:
from pyspark.ml.evaluation import RegressionEvaluator

def evaluate_model(predictions, model_name):
    evaluators = {
        'MAE': RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='mae'),
        'RMSE': RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='rmse'),
        'R2': RegressionEvaluator(labelCol='label', predictionCol='prediction', metricName='r2')
    }
    print(f'=== {model_name} ===')
    results = {}
    for name, evaluator in evaluators.items():
        value = evaluator.evaluate(predictions)
        results[name] = value
        print(f'  {name}: {value:,.2f}')
    return results

gbt_metrics = evaluate_model(gbt_predictions, 'Gradient-Boosted Tree')
print()
rf_metrics = evaluate_model(rf_predictions, 'Random Forest')

=== Gradient-Boosted Tree ===
  MAE: 7,176.49
  RMSE: 13,313.92
  R2: 0.69

=== Random Forest ===
  MAE: 7,430.19
  RMSE: 14,098.41
  R2: 0.65


### 6.3 Visualisasi Prediksi vs Aktual

In [30]:
# Pilih model terbaik berdasarkan MAE
if gbt_metrics['MAE'] <= rf_metrics['MAE']:
    best_predictions = gbt_predictions
    best_name = 'Gradient-Boosted Tree'
else:
    best_predictions = rf_predictions
    best_name = 'Random Forest'

print(f'Model terbaik: {best_name}')

pred_pd = (
    best_predictions
    .select('super_category', 'year', 'month', 'label', 'prediction')
    .toPandas()
)
pred_pd['period'] = pred_pd['year'].astype(str) + '-' + pred_pd['month'].astype(str).str.zfill(2)

Model terbaik: Gradient-Boosted Tree


In [31]:
# Prediksi vs Aktual -- agregat bulanan (semua kategori)
agg_pred = pred_pd.groupby('period')[['label', 'prediction']].sum().reset_index().sort_values('period')

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(agg_pred['period'], agg_pred['label'] / 1e6, 'o-', label='Aktual', linewidth=2)
ax.plot(agg_pred['period'], agg_pred['prediction'] / 1e6, 's--', label='Prediksi', linewidth=2)
ax.set_xlabel('Periode')
ax.set_ylabel('Volume (Juta Liter)')
ax.set_title(f'Prediksi vs Aktual -- Total Volume Bulanan ({best_name})')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('eval_pred_vs_actual.png', bbox_inches='tight')
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_5956\3078888150.py:14: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [32]:
# Prediksi vs Aktual per kategori utama (top 4)
top4 = pred_pd.groupby('super_category')['label'].sum().nlargest(4).index.tolist()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, cat in zip(axes.flatten(), top4):
    subset = pred_pd[pred_pd['super_category'] == cat].sort_values('period')
    ax.plot(subset['period'], subset['label'] / 1e3, 'o-', label='Aktual', markersize=4)
    ax.plot(subset['period'], subset['prediction'] / 1e3, 's--', label='Prediksi', markersize=4)
    ax.set_title(cat)
    ax.set_ylabel('Volume (Ribu Liter)')
    ax.legend(fontsize=8)
    ax.tick_params(axis='x', rotation=45, labelsize=7)

plt.suptitle(f'Prediksi vs Aktual per Kategori ({best_name})', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('eval_per_category.png', bbox_inches='tight')
plt.show()

C:\Users\user\AppData\Local\Temp\ipykernel_5956\4105383794.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### 6.4 Feature Importance

In [33]:
import numpy as np

if best_name == 'Gradient-Boosted Tree':
    importances = gbt_model.featureImportances.toArray()
else:
    importances = rf_model.featureImportances.toArray()

# Buat nama fitur otomatis berdasarkan jumlah importances aktual
# Tidak perlu tebak-tebakan jumlah slot OHE
input_cols_no_ohe = [c for c in assembler.getInputCols() if c != 'category_ohe']
ohe_slots = len(importances) - len(input_cols_no_ohe)
ohe_names = [f'category_{l}' for l in indexer_model.labels[:ohe_slots]]

feature_names = input_cols_no_ohe + ohe_names

print(f'Non-OHE features : {len(input_cols_no_ohe)}')
print(f'OHE slots aktual : {ohe_slots}')
print(f'Total feature_names: {len(feature_names)}')
print(f'Importances count  : {len(importances)}')

assert len(feature_names) == len(importances), \
    f'Masih mismatch: {len(feature_names)} vs {len(importances)}'

# Plot
fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(fi_df['feature'][::-1], fi_df['importance'][::-1],
        color=sns.color_palette('crest', len(fi_df)))
ax.set_xlabel('Importance')
ax.set_title(f'Top 15 Feature Importance ({best_name})')
plt.tight_layout()
plt.savefig('eval_feature_importance.png', bbox_inches='tight')
plt.show()

Non-OHE features : 23
OHE slots aktual : 18
Total feature_names: 41
Importances count  : 41


C:\Users\user\AppData\Local\Temp\ipykernel_5956\904847087.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Langkah 7: Interpretasi Bisnis & Rekomendasi

In [34]:
# Ringkasan metrik akhir
print('=' * 60)
print('RINGKASAN EVALUASI MODEL')
print('=' * 60)
print(f'Model Terpilih: {best_name}')
best_metrics = gbt_metrics if best_name == 'Gradient-Boosted Tree' else rf_metrics
print(f'MAE:  {best_metrics["MAE"]:,.2f} Liter')
print(f'RMSE: {best_metrics["RMSE"]:,.2f} Liter')
print(f'R2:   {best_metrics["R2"]:.4f}')
print('=' * 60)
print(f'Safety Stock (95% SL): {best_metrics["MAE"] * 1.65:,.0f} Liter per kategori/bulan')
print(f'Safety Stock (99% SL): {best_metrics["MAE"] * 2.33:,.0f} Liter per kategori/bulan')

RINGKASAN EVALUASI MODEL
Model Terpilih: Gradient-Boosted Tree
MAE:  7,176.49 Liter
RMSE: 13,313.92 Liter
R2:   0.6857
Safety Stock (95% SL): 11,841 Liter per kategori/bulan
Safety Stock (99% SL): 16,721 Liter per kategori/bulan


## Interpretasi Bisnis: Penggunaan MAE sebagai Acuan *Safety Stock*

---

### 📊 Apa itu MAE dalam Konteks Ini?

**Mean Absolute Error (MAE)** mengukur rata-rata **selisih absolut** antara volume yang diprediksi model dengan volume yang benar-benar terjual, dalam satuan **Liter per kategori per bulan**.  
Secara sederhana, MAE menjawab pertanyaan: *"Seberapa besar rata-rata 'meleset'-nya prediksi kita?"*

---

### 🏭 Aplikasi Langsung untuk Manajer Gudang Iowa

Model ini menghasilkan prediksi volume penjualan per kategori minuman keras setiap bulan.  
Meskipun prediksi tidak pernah sempurna, nilai MAE memberikan **batas ketidakpastian yang terukur** — sebuah angka konkret yang dapat langsung diterjemahkan menjadi kebijakan stok.

**Formula Safety Stock berbasis MAE:**

```
Safety Stock = Z × MAE
```

| Service Level | Faktor Z | Interpretasi |
|:---:|:---:|:---|
| 90% | 1.28 | Risiko stockout 10% — cocok untuk kategori volume rendah |
| 95% | 1.65 | Risiko stockout 5% — **rekomendasi untuk kategori menengah** |
| 99% | 2.33 | Risiko stockout 1% — **wajib untuk VODKA & WHISKEY (volume dominan)** |

---

### 📦 Langkah Operasional yang Direkomendasikan

1. **Hitung Prediksi Bulanan:** Jalankan model setiap awal bulan untuk mendapatkan `predicted_volume` per kategori.
2. **Tentukan Safety Stock per Kategori:** Gunakan formula `Z × MAE`; pilih Z sesuai kepentingan kategori.
3. **Tetapkan Stok Minimum (Reorder Point):**  
   `Stok Minimum = Predicted Volume + Safety Stock`
4. **Prioritaskan Kategori Kritis:** Berdasarkan EDA, **VODKA** dan **BOURBON** memiliki volume tertinggi — kesalahan prediksi sekecil apapun berdampak besar. Terapkan safety level 99% untuk kategori ini.
5. **Review Bulanan:** Jika error model di bulan berjalan melebihi MAE historis lebih dari 2×, tandai sebagai anomali dan lakukan pengecekan manual (holiday tidak tercapture, supply disruption, dll).

---

### ⚠️ Keterbatasan & Catatan

- MAE dihitung sebagai **rata-rata** dari semua kategori; kategori dengan variabilitas tinggi (seperti **TEQUILA** dan **GIN** yang bersifat musiman) mungkin membutuhkan safety stock lebih besar dari formula standar.
- Model perlu di-retrain secara berkala (minimal setiap kuartal) untuk menangkap perubahan tren pasar.
- Safety Stock ini hanya merupakan **floor** (batas bawah stok minimum) — Manajer tetap perlu mempertimbangkan lead time pengiriman dari distributor ke gudang pusat Iowa.


In [35]:
spark.stop()
print('Spark session stopped.')

Spark session stopped.
